# ⚡ NeuralOps Notebook 38: Weight Ternarization and 1-Bit Architectures
## Chapter 26: Ternary Weights {-1, 0, +1}, BitNet b1.58 Scaling, and Multiplication-Free Inference

> 💡 *"Floating-point multiplications dominate accelerator energy consumption. Ternary architectures constrain weights strictly to {-1, 0, +1}, eliminating silicon multipliers in favor of pure integer addition and subtraction."*

---

### 1. Pedagogical Overview & Industrial Context
BitNet b1.58 and ternary neural networks quantize every weight to 1.58 bits ($\log_2 3$ states):
$$\mathbf{W} \in \{-1, \; 0, \; +1\}^{D_{\mathrm{out}} \times D_{\mathrm{in}}}$$
During inference, the inner product $\mathbf{y} = \mathbf{W} \mathbf{x}$ requires **zero floating-point multiplications**: an entry of $+1$ adds activation $x_j$, $-1$ subtracts $x_j$, and $0$ omits it. This slashes energy consumption by up to $10\times$ and DRAM bandwidth by up to $10\times$ over FP16.

In this tutorial, you will explore:
1. Mathematical formulation of symmetric ternary quantization and thresholding $\Delta$.
2. Straight-Through Estimator (STE) autograd gradient propagation.
3. PyTorch reference implementation with activation scaling.
4. 4-panel publication-grade diagnostic figure.


---

## 2. Mathematical Formalism & Ternarization Invariants

Given full-precision latent weights $\mathbf{W} \in \mathbb{R}^{M \times K}$:

### 1. Scaling Factor & Threshold:
$$\gamma = \frac{1}{M K} \sum_{i, j} |W_{i, j}|, \quad \Delta = 0.7 \cdot \gamma$$

### 2. Ternary Quantization Function:
$$\widehat{W}_{i, j} = \begin{cases} +1 & \text{if } W_{i, j} > +\Delta \\ 0 & \text{if } |W_{i, j}| \le \Delta \\ -1 & \text{if } W_{i, j} < -\Delta \end{cases}$$

### 3. Forward MatMul:
$$\mathbf{y} = \left( \mathbf{x} \cdot \widehat{\mathbf{W}}^T \right) \cdot \beta, \quad \beta = \frac{\sum |W_{i, j}|}{\sum |\widehat{W}_{i, j}|}$$


In [ ]:
import os, sys, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


---

## 3. Pure Vectorized Reference Implementation (Level 3 Fallback)

We implement a TernaryLinear layer using the Straight-Through Estimator (STE).


In [ ]:
class TernaryQuantizer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, w: torch.Tensor, delta_factor: float = 0.7) -> tuple[torch.Tensor, torch.Tensor]:
        ctx.save_for_backward(w)
        gamma = w.abs().mean()
        delta = delta_factor * gamma
        w_ternary = torch.zeros_like(w)
        w_ternary = torch.where(w > delta, torch.ones_like(w), w_ternary)
        w_ternary = torch.where(w < -delta, -torch.ones_like(w), w_ternary)
        active_count = (w_ternary != 0).float().sum().clamp(min=1.0)
        alpha = w.abs().sum() / active_count
        return w_ternary, alpha

    @staticmethod
    def backward(ctx, grad_w_ternary, grad_alpha):
        w, = ctx.saved_tensors
        # STE: clip gradients outside [-1, 1]
        grad_w = grad_w_ternary.clone()
        grad_w = torch.where(w.abs() <= 1.0, grad_w, torch.zeros_like(grad_w))
        return grad_w, None

class BitNet158Linear(nn.Module):
    def __init__(self, in_features=128, out_features=128):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(out_features, in_features) * 0.05)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        w_t, alpha = TernaryQuantizer.apply(self.weight)
        return F.linear(x, w_t * alpha)

ternary_linear = BitNet158Linear(128, 128).to(device)
x_in = torch.randn(4, 128, device=device)
y_out = ternary_linear(x_in)
print(f'Ternary Forward Output Shape: {y_out.shape}')


---

## 4. Standalone 4-Panel Publication-Grade Diagnostics

We visualize the ternary weight distribution {-1, 0, +1}, energy consumption, autograd gradient flow, and memory footprint.


In [ ]:
with torch.no_grad():
    w_t, alpha = TernaryQuantizer.apply(ternary_linear.weight)
    vals = w_t.cpu().numpy().flatten()

fig, axes = plt.subplots(2, 2, figsize=(14, 10), dpi=100)

# Panel 1: Ternary State Distribution {-1, 0, +1}
ax1 = axes[0, 0]
counts = [np.sum(vals == -1), np.sum(vals == 0), np.sum(vals == 1)]
ax1.bar(['-1 (Negative)', '0 (Pruned)', '+1 (Positive)'], counts,
        color=['#d9534f', '#6c757d', '#5cb85c'], edgecolor='black', alpha=0.85)
ax1.set_title('BitNet b1.58 Ternary Weights State Histogram', fontsize=12, fontweight='bold')
ax1.set_ylabel('Weight Element Count', fontsize=10)
ax1.grid(True, axis='y', linestyle='--', alpha=0.5)

# Panel 2: Energy Consumption Comparison
ax2 = axes[0, 1]
precisions = ['FP32 Mul', 'FP16 Mul', 'INT8 Mul', 'Ternary Add (b1.58)']
energy_pj = [3.7, 1.1, 0.2, 0.03] # picojoules per op
ax2.barh(precisions, energy_pj, color=['#d9534f', '#f0ad4e', '#0275d8', '#5cb85c'], edgecolor='black', alpha=0.85)
ax2.set_title('Arithmetic Operation Energy (pJ / Op)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Energy per Operation (picojoules, log scale)', fontsize=10)
ax2.set_xscale('log')
ax2.grid(True, axis='x', linestyle='--', alpha=0.5)

# Panel 3: Autograd Gradient Flow (STE)
ax3 = axes[1, 0]
x_g = torch.randn(4, 128, device=device, requires_grad=True)
loss = ternary_linear(x_g).pow(2).mean()
loss.backward()
g_vals = ternary_linear.weight.grad.cpu().numpy().flatten()
ax3.hist(g_vals, bins=30, color='#6f42c1', edgecolor='black', alpha=0.85)
ax3.set_title('STE Latent Weight Gradient Distribution', fontsize=12, fontweight='bold')
ax3.set_xlabel('Gradient Magnitude', fontsize=10)
ax3.set_ylabel('Count', fontsize=10)
ax3.grid(True, linestyle='--', alpha=0.5)

# Panel 4: Model Weight DRAM Footprint
ax4 = axes[1, 1]
models = ['7B FP16', '7B INT8', '7B INT4', '7B BitNet b1.58']
vram_gb = [14.0, 7.0, 3.5, 1.38]
ax4.bar(models, vram_gb, color=['#d9534f', '#0275d8', '#20c997', '#5cb85c'], edgecolor='black', alpha=0.85)
ax4.set_title('7B Model Weights DRAM Footprint (GB)', fontsize=12, fontweight='bold')
ax4.set_ylabel('VRAM Size (GB)', fontsize=10)
ax4.grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
os.makedirs('results/plots', exist_ok=True)
plt.savefig('results/plots/38_weight_ternarization_and_1bit_architectures.png', dpi=300)
print('✓ Figure saved to results/plots/38_weight_ternarization_and_1bit_architectures.png')
plt.show()


---

## 5. Industrial Takeaways & Deployment Standards

1. **Multiplication-Free Acceleration:** Matrix multiplies transform into pure parallel integer addition trees.
2. **Optimal Threshold $\Delta = 0.7 \cdot \mathbb{E}[|W|]$:** Provides the theoretical maximum entropy for ternary coding.
3. **Zero-Fake Export Compatibility:** Pack 2-bit ternary states into `int8` containers with scaling scalars.
